# 04 - Models: Deep Neural Network (DNN) con TF-IDF (SVD)

Este notebook entrena un modelo **Deep Neural Network (DNN)** usando la codificacion **TF-IDF**. Para que corra rápido, aplica **TruncatedSVD** previamente. Todo el flujo se registra en MLflow.

In [1]:
!pip install -q torch scikit-learn scipy joblib matplotlib seaborn mlflow

import copy, json, logging, os, warnings
from pathlib import Path

import joblib, mlflow, mlflow.pytorch
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn

from scipy.sparse import load_npz
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score, precision_score, recall_score, roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset

SEED       = 42
MEMBER     = "asantiagodiaz"
MLFLOW_URI = "http://ec2-52-5-36-177.compute-1.amazonaws.com:5000"

os.chdir('/home/ec2-user/SageMaker')

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
logging.getLogger('mlflow.tracking.request_header.registry').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')
print('Device:', DEVICE)

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device: cuda


In [2]:
X_tr          = load_npz('X_tr.npz')
X_val         = load_npz('X_val.npz')
X_train_tfidf = load_npz('X_train_tfidf.npz')
X_test_tfidf  = load_npz('X_test_tfidf.npz')

y_tr    = joblib.load('y_tr.pkl')
y_val   = joblib.load('y_val.pkl')
y_train = joblib.load('y_train.pkl')
y_test  = joblib.load('y_test.pkl')

# ── MUESTREO PARA TUNING (100k) ────────────────────────────────
idx_tune, _ = train_test_split(np.arange(X_tr.shape[0]), train_size=100000, random_state=SEED, stratify=y_tr)
X_tr_tune = X_tr[idx_tune]
y_tr_tune = np.asarray(y_tr)[idx_tune]

# ── REDUCCION DE DIMENSIONALIDAD SVD ─────────────────────────
print("Aplicando SVD para acelerar DNN...")
N_COMPONENTS = 300
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=SEED)

X_tr_tune_d   = svd.fit_transform(X_tr_tune).astype(np.float32)
X_val_d       = svd.transform(X_val).astype(np.float32)
X_train_final_d = svd.transform(X_train_tfidf).astype(np.float32)
X_test_final_d  = svd.transform(X_test_tfidf).astype(np.float32)
print("SVD completado.")

Aplicando SVD para acelerar DNN...
SVD completado.


In [3]:
class DenseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

def make_loader(X, y, batch_size, shuffle=False):
    return DataLoader(DenseDataset(X, y), batch_size=batch_size, shuffle=shuffle)

class SentimentDNN(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for d in hidden_dims:
            layers.extend([nn.Linear(prev_dim, d), nn.ReLU(), nn.Dropout(dropout)])
            prev_dim = d
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def predict_proba(model, loader, device):
    model.eval()
    probs = []
    with torch.no_grad():
        for X_b, _ in loader:
            probs.append(torch.sigmoid(model(X_b.to(device))).cpu().numpy().ravel())
    return np.concatenate(probs)

def compute_metrics(y_true, y_proba):
    yp = (y_proba >= 0.5).astype(int)
    return {
        'accuracy': round(accuracy_score(y_true, yp), 4),
        'f1_macro': round(f1_score(y_true, yp, average='macro'), 4),
        'roc_auc': round(roc_auc_score(y_true, y_proba), 4)
    }

def fit_dnn(config, X_t, y_t, X_v, y_v, device):
    train_loader = make_loader(X_t, y_t, config['batch_size'], True)
    eval_loader  = make_loader(X_v, y_v, config['batch_size'])
    model = SentimentDNN(X_t.shape[1], config['hidden_dims'], config['dropout']).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=config['lr'])
    crit = nn.BCEWithLogitsLoss()
    best_f1, best_ep, best_state, pat = -1, 0, None, config['patience']
    
    for ep in range(1, config['epochs'] + 1):
        model.train()
        for X_b, y_b in train_loader:
            opt.zero_grad()
            loss = crit(model(X_b.to(device)), y_b.to(device))
            loss.backward(); opt.step()
        
        metrics = compute_metrics(y_v, predict_proba(model, eval_loader, device))
        if metrics['f1_macro'] > best_f1:
            best_f1, best_ep, best_state, pat = metrics['f1_macro'], ep, copy.deepcopy(model.state_dict()), config['patience']
        elif (pat := pat - 1) == 0: break
            
    model.load_state_dict(best_state)
    return model, best_ep

In [4]:
param_grid = [
    {'hidden_dims': [128, 64], 'dropout': 0.30, 'lr': 1e-3, 'batch_size': 1024, 'epochs': 5, 'patience': 2},
    {'hidden_dims': [256, 64], 'dropout': 0.35, 'lr': 8e-4, 'batch_size': 1024, 'epochs': 5, 'patience': 2},
]

results_dnn, best_f1, best_params, best_model, best_epoch = [], -1, None, None, None

for i, p in enumerate(param_grid, 1):
    print(f'Entrenando cfg {i}/{len(param_grid)}: {p}')
    model, ep = fit_dnn(p, X_tr_tune_d, y_tr_tune, X_val_d, y_val, DEVICE)
    m = compute_metrics(y_val, predict_proba(model, make_loader(X_val_d, y_val, p['batch_size']), DEVICE))
    results_dnn.append({**p, 'best_ep': ep, **m})
    if m['f1_macro'] > best_f1:
        best_f1, best_params, best_model, best_epoch = m['f1_macro'], p, model, ep

display(pd.DataFrame(results_dnn).sort_values('f1_macro', ascending=False))
print('Best params:', best_params)

Entrenando cfg 1/2: {'hidden_dims': [128, 64], 'dropout': 0.3, 'lr': 0.001, 'batch_size': 1024, 'epochs': 5, 'patience': 2}
Entrenando cfg 2/2: {'hidden_dims': [256, 64], 'dropout': 0.35, 'lr': 0.0008, 'batch_size': 1024, 'epochs': 5, 'patience': 2}


,hidden_dims,dropout,lr,batch_size,epochs,patience,best_ep,accuracy,f1_macro,roc_auc
0,"[128, 64]",0.30,0.0010,1024,5,2,5,0.7529,0.7527,0.8332
1,"[256, 64]",0.35,0.0008,1024,5,2,5,0.7519,0.7519,0.8324


Best params: {'hidden_dims': [128, 64], 'dropout': 0.3, 'lr': 0.001, 'batch_size': 1024, 'epochs': 5, 'patience': 2}


In [5]:
print('\nEntrenando modelo final...')
final_p = {**best_params, 'epochs': best_epoch, 'patience': best_epoch + 1}
model_final, _ = fit_dnn(final_p, X_train_final_d, y_train, X_val_d, y_val, DEVICE)

y_proba = predict_proba(model_final, make_loader(X_test_final_d, y_test, final_p['batch_size']), DEVICE)
y_pred = (y_proba >= 0.5).astype(int)
metrics_test = compute_metrics(y_test, y_proba)
print('\nMetricas Test:', metrics_test)


Entrenando modelo final...

Metricas Test: {'accuracy': 0.7773, 'f1_macro': 0.7773, 'roc_auc': 0.8608}


In [6]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('Parcial_1_NLP')

with mlflow.start_run(run_name=f'DNN_TFIDF-SVD_{MEMBER}') as run:
    mlflow.set_tags({
        'user': 'Santiago Diaz', 'member': MEMBER,
        'model_type': 'DeepNeuralNetwork',
        'encoding': 'TF-IDF+SVD', 'dataset': 'Sentiment140Twitter',
    })
    
    mlflow.log_params({
        'prep_remove_urls': True, 'prep_remove_mentions': True,
        'prep_remove_hashtags': True, 'prep_remove_emojis': True,
        'prep_remove_stopwords': False, 'prep_lemmatization': False,
    })
    
    mlflow.log_params({
        'vec_type': 'TfidfVectorizer', 'vec_max_features': 50000,
        'vec_min_df': 5, 'vec_max_df': 0.95,
        'vec_ngram_range': '(1,1)',
        'vec_sublinear_tf': False, 'vec_norm': 'None',
    })
    
    mlflow.log_params({
        'model': 'DNN', 'dim_reduction': 'TruncatedSVD',
        'n_components': N_COMPONENTS,
        'hidden_dims': str(best_params['hidden_dims']),
        'dropout': best_params['dropout'], 'lr': best_params['lr'],
        'batch_size': best_params['batch_size'], 'best_epoch': best_epoch,
        'train_size_tuning': len(y_tr_tune), 'val_size': len(y_val),
        'final_train_size': len(y_train),
        'test_size': len(y_test), 'vocab_size': X_train_tfidf.shape[1],
        'device': str(DEVICE),
    })

    # Train metrics — full dataset sobre la versión reducida (SVD)
    train_loader_eval = make_loader(X_train_final_d, y_train, batch_size=best_params['batch_size'])
    y_proba_train = predict_proba(model_final, train_loader_eval, DEVICE)
    y_pred_train  = (y_proba_train >= 0.5).astype(int)
    mlflow.log_metrics({
        'train_accuracy':  round(accuracy_score(y_train, y_pred_train),                    4),
        'train_f1_macro':  round(f1_score(y_train,       y_pred_train, average='macro'),   4),
        'train_precision': round(precision_score(y_train, y_pred_train, average='macro'),  4),
        'train_recall':    round(recall_score(y_train,    y_pred_train, average='macro'),  4),
        'train_roc_auc':   round(roc_auc_score(y_train,  y_proba_train),                  4),
    })

    # Val metrics — sobre la versión reducida (SVD)
    val_loader_eval = make_loader(X_val_d, y_val, batch_size=best_params['batch_size'])
    y_proba_val = predict_proba(best_model, val_loader_eval, DEVICE)
    y_pred_val  = (y_proba_val >= 0.5).astype(int)
    mlflow.log_metrics({
        'val_accuracy':  round(accuracy_score(y_val, y_pred_val),                    4),
        'val_f1_macro':  round(f1_score(y_val,       y_pred_val, average='macro'),   4),
        'val_precision': round(precision_score(y_val, y_pred_val, average='macro'),  4),
        'val_recall':    round(recall_score(y_val,    y_pred_val, average='macro'),  4),
        'val_roc_auc':   round(roc_auc_score(y_val,  y_proba_val),                  4),
    })

    # Test metrics — usando el diccionario metrics_test que ya tiene todo
    mlflow.log_metrics({
        'test_accuracy':  metrics_test['accuracy'],
        'test_f1_macro':  metrics_test['f1_macro'],
        'test_roc_auc':   metrics_test['roc_auc'],
    })

    # Como en este código resumido quitamos las líneas que guardan el CSV/config para que fuera más rápido,
    # solo guardamos el modelo de pytorch. Si vuelves a agregar la celda que crea esos archivos, 
    # puedes descomentar estas dos líneas:
    # mlflow.log_artifact('results_dnn_tfidf.csv')
    # mlflow.log_artifact('dnn_tfidf_config.json')
    
    mlflow.pytorch.log_model(
        model_final.cpu(),
        artifact_path='model',
        registered_model_name=f'DNN_TFIDF-SVD_{MEMBER}'
    )
    model_final.to(DEVICE)

    print('=' * 55)
    print(f'  Run ID   : {run.info.run_id}')
    print(f"  Test F1  : {metrics_test['f1_macro']:.4f}")
    print(f"  Test AUC : {metrics_test['roc_auc']:.4f}")
    print('=' * 55)


2026/03/10 00:43:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 00:43:14 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/10 00:43:14 WARNING mlflow.utils.requirements_utils: Found torch version (2.6.0+cu124) contains a local version label (+cu124). MLflow logged a pip requirement for this package as 'torch==2.6.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/10 00:43:18 WARNING mlflow.utils.requirements_utils: Found torch version (2.6.0+cu124) contains a local version label (+cu124). ML

  Run ID   : d3532ee01a534921aea0b2310252d464
  Test F1  : 0.7773
  Test AUC : 0.8608
🏃 View run DNN_TFIDF-SVD_asantiagodiaz at: http://ec2-52-5-36-177.compute-1.amazonaws.com:5000/#/experiments/1/runs/d3532ee01a534921aea0b2310252d464
🧪 View experiment at: http://ec2-52-5-36-177.compute-1.amazonaws.com:5000/#/experiments/1
